# MatForge — Notebook 03: Training
#
# Complete training pipeline for MatForge (PVT-v2-B1 + FPN decoder + three independent heads).
#
# **Before running**: verify that `matforge_split.csv` is present in the dataset's
# `/relabeling/` folder. If absent, the notebook regenerates it with SEED=42.
#
# **Execution modes** (set in Cell 2):
# - `DRY_RUN = True`  → 5 epochs, validate every epoch, verbose diagnostics.
# - `DRY_RUN = False` → 90 epochs, validate every 5 epochs, clean output.
#
# Run all cells in order. The training loop is resumable from any checkpoint.

## Cell 1 — Package installation

In [1]:
import subprocess
subprocess.run(["pip", "install", "lpips", "--quiet"], check=True)
print("lpips ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.1 MB/s eta 0:00:00
lpips ready.


## Cell 2 — Imports and configuration

In [2]:
import os, json, math, time, random, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, BatchSampler
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
from torchvision.transforms import functional as TF
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import timm
import lpips as lpips_lib
 
# ─── Execution mode ──────────────────────────────────────────────────────────
DRY_RUN = False   # flip to False for the full 90-epoch run
 
# ─── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
 
# ─── Training hyperparameters ─────────────────────────────────────────────────
TOTAL_EPOCHS = 5 if DRY_RUN else 90  # supervised epochs (already done)
GAN_ACTIVE       = True        # set False to skip GAN entirely
GAN_EPOCHS       = 20          # additional epochs of GAN fine-tuning
GAN_TOTAL    = 3 if DRY_RUN else GAN_EPOCHS
BATCH_SIZE       = 8
LR_ENCODER       = 1e-4
LR_DECODER       = 3e-4
# LRs for cosine restart when resuming — 30% of original
LR_ENCODER_RESUME = 2e-5
LR_DECODER_RESUME = 6e-5
WEIGHT_DECAY     = 1e-2
GRAD_CLIP        = 1.0
WARMUP_EPOCHS    = 5
FREEZE_EPOCHS    = 3           # encoder frozen for this many epochs at start
EMA_DECAY        = 0.999       # activated when encoder unfreezes
VALIDATE_EVERY   = 1 if DRY_RUN else 5
SAVE_CKPT_EVERY  = 5           # model-only checkpoint; always every 5 epochs
NUM_WORKERS      = 2
 
# ─── Curriculum resolution ────────────────────────────────────────────────────
CURRICULUM_EPOCH = 65          # switch crop size at this epoch
CROP_PHASE1      = 256         # epochs 0 → CURRICULUM_EPOCH-1
CROP_PHASE2      = 320         # epochs CURRICULUM_EPOCH → end; skip in DRY_RUN
 
# ─── Loss weights (fixed) ─────────────────────────────────────────────────────
W_NORMAL    = 1.0
W_ROUGHNESS = 0.8
W_METALLIC  = 0.5
W_GRAD      = 0.15
METALLIC_POS_WEIGHT = 8.0
 
# ─── L_render progressive activation schedule (epoch → weights) ───────────────
# Key = (epoch_start_inclusive, epoch_end_exclusive), value = weight
RENDER_L1_SCHED   = {(0, 5): 0.00, (5, 15): 0.10, (15, 999): 0.15}
RENDER_LPIPS_SCHED = {(0, 16): 0.00, (16, 999): 0.03}
 
def get_render_weights(epoch: int) -> tuple:
    """Returns (w_L1, w_LPIPS) for the current epoch from the schedule tables."""
    def lookup(sched):
        for (s, e), w in sched.items():
            if s <= epoch < e:
                return w
        return 0.0
    return lookup(RENDER_L1_SCHED), lookup(RENDER_LPIPS_SCHED)

# ─── GAN fine-tuning hyperparameters ──────────────────────────────────────────
LR_D             = 1e-4        # discriminator learning rate
R1_INTERVAL      = 16          # lazy R1: compute penalty every N steps
R1_LAMBDA        = 10.0        # R1 gradient penalty weight
W_FM             = 10.0        # feature matching loss weight (stable signal)
# Progressive adversarial weight schedule: epoch_gan → w_adv
GAN_ADV_SCHED    = {(0,5): 0.02, (5,10): 0.05, (10,999): 0.10}

def get_adv_weight(epoch_gan: int) -> float:
    for (s, e), w in GAN_ADV_SCHED.items():
        if s <= epoch_gan < e:
            return w
    return 0.10
 
# ─── Sampler: guaranteed metal per batch ──────────────────────────────────────
N_METAL_PER_BATCH = 2
 
# ─── Reference textures ───────────────────────────────────────────────────────
REFERENCE_TEXTURE = "metal_0081.png"   # special hard-case reference
GROUPS = [
    "brick_terracotta", "ceramic_ground", "concrete_plaster",
    "marble_smooth", "metal", "mixed_ambiguous", "stone_rough", "wood",
]
 
# ─── Column names (match relabeling_final.csv) ────────────────────────────────
COL_FILENAME = "filename"
COL_GROUP    = "functional_group"
 
# ─── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT  = Path("/kaggle/input/datasets/mjgut05/matforge-pbr-dataset/matforge_dataset")
RGB_DIR       = DATASET_ROOT / "maps" / "rgb"
NORMAL_DIR    = DATASET_ROOT / "maps" / "normal"
ROUGHNESS_DIR = DATASET_ROOT / "maps" / "roughness"
METALLIC_DIR  = DATASET_ROOT / "maps" / "metallic"
RELABELING_CSV = DATASET_ROOT / "relabeling" / "relabeling_final.csv"
SPLIT_CSV      = Path("/kaggle/input/datasets/mjgut05/matforge-pbr-dataset") / "matforge_split.csv"

# For GAN fine-tuning: load supervised checkpoints from the UPDATED ep20 dataset
GAN_CKPT_SOURCE = Path("/kaggle/input/matforge-checkpoints-ep20/checkpoints")
 
OUTPUT_DIR = Path("/kaggle/working")
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
PANELS_DIR = OUTPUT_DIR / "panels"
CKPT_DIR.mkdir(exist_ok=True)
PANELS_DIR.mkdir(exist_ok=True)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
 
print(f"Mode   : {'DRY RUN (5 epochs)' if DRY_RUN else 'FULL TRAINING (90 epochs)'}")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mode   : FULL TRAINING (90 epochs)
Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## Cell 3 — Data: split, augmentation, dataset, sampler, loaders

In [3]:
# ─── Augmentation ─────────────────────────────────────────────────────────────
 
def apply_geometric_augmentation(rgb, normal, roughness, metallic,
                                  p_hflip=0.5, p_vflip=0.5):
    """
    Applies spatially coherent geometric augmentations to all four maps.
 
    Normal maps store surface orientation as (X, Y, Z) in tangent space (OpenGL,
    +Z outward). A purely spatial flip without the matching component sign change
    produces a physically incorrect map. The rules below maintain consistency:
 
        Horizontal flip  : spatial hflip + negate X channel
        Vertical flip    : spatial vflip + negate Y channel
        Rotation 90° CCW : spatial rot90(k=1) + (X,Y) → (-Y, X)
        Rotation 180°    : spatial rot90(k=2) + (X,Y) → (-X,-Y)
        Rotation 270° CCW: spatial rot90(k=3) + (X,Y) → ( Y,-X)
 
    Rotations are restricted to multiples of 90° to avoid bilinear artefacts
    on fine detail and to respect directional anisotropy in materials like wood.
    """
    if torch.rand(1).item() < p_hflip:
        rgb = TF.hflip(rgb); roughness = TF.hflip(roughness)
        metallic = TF.hflip(metallic); normal = TF.hflip(normal)
        normal = torch.cat([-normal[0:1], normal[1:2], normal[2:3]], dim=0)
 
    if torch.rand(1).item() < p_vflip:
        rgb = TF.vflip(rgb); roughness = TF.vflip(roughness)
        metallic = TF.vflip(metallic); normal = TF.vflip(normal)
        normal = torch.cat([normal[0:1], -normal[1:2], normal[2:3]], dim=0)
 
    k = torch.randint(0, 4, (1,)).item()
    if k > 0:
        rgb = torch.rot90(rgb, k, dims=[1, 2])
        roughness = torch.rot90(roughness, k, dims=[1, 2])
        metallic  = torch.rot90(metallic,  k, dims=[1, 2])
        normal    = torch.rot90(normal,    k, dims=[1, 2])
        x, y, z   = normal[0:1], normal[1:2], normal[2:3]
        if k == 1: normal = torch.cat([-y,  x, z], dim=0)
        elif k == 2: normal = torch.cat([-x, -y, z], dim=0)
        elif k == 3: normal = torch.cat([ y, -x, z], dim=0)
 
    return rgb, normal, roughness, metallic
 
 
def apply_photometric_augmentation(rgb):
    """
    Mild colour jitter on the RGB input only. GT maps are never perturbed.
 
    Conservative ranges prevent introducing input-GT pairs that are physically
    inconsistent: the roughness and normal GTs are fixed while the colour
    shifts, which would teach the model spurious colour-roughness correlations
    if the jitter were too large.
    """
    pil = TF.to_pil_image(rgb)
    pil = transforms.ColorJitter(
        brightness=0.08, contrast=0.08, saturation=0.05, hue=0.02
    )(pil)
    return TF.to_tensor(pil)
 
 
# ─── Dataset ──────────────────────────────────────────────────────────────────
 
class MatForgeDataset(Dataset):
    """
    MatForge PBR texture dataset.
 
    Returns per-sample dict with keys:
        rgb       (3,H,W) float32 ImageNet-normalised
        normal    (3,H,W) float32 [-1,1]
        roughness (1,H,W) float32 [0,1]
        metallic  (1,H,W) float32 [0,1]
        group     str
 
    Metallic is loaded from disk for the 'metal' group only; for all other
    groups a zero tensor is constructed on-the-fly. Zero is the correct
    default for dielectric materials in the UE4 metallic/roughness workflow.
 
    Normal maps are stored as (N+1)/2 in PNG [0,255] and remapped to [-1,1].
 
    Input normalisation uses ImageNet statistics because PVT-v2-B1 is pretrained
    on ImageNet-1K. Changing to dataset-specific statistics would corrupt the
    pretrained feature representations.
    """
    _MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    _STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
 
    def __init__(self, df, split, crop_size=256, augment=True):
        assert split in ("train", "val")
        self.df        = df[df["split"] == split].reset_index(drop=True)
        self.crop_size = crop_size
        self.augment   = augment and (split == "train")
 
    def __len__(self): return len(self.df)
 
    @staticmethod
    def _load(path, mode):
        return TF.to_tensor(Image.open(path).convert(mode))
 
    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        fname    = str(row[COL_FILENAME])
        group    = str(row[COL_GROUP])
 
        rgb       = self._load(RGB_DIR       / fname, "RGB")
        normal_01 = self._load(NORMAL_DIR    / fname, "RGB")
        roughness = self._load(ROUGHNESS_DIR / fname, "L")
 
        metal_path = METALLIC_DIR / fname
        if group == "metal" and metal_path.exists():
            metallic = self._load(metal_path, "L")
        else:
            metallic = torch.zeros(1, rgb.shape[1], rgb.shape[2])
 
        normal = normal_01 * 2.0 - 1.0
 
        i, j, h, w = transforms.RandomCrop.get_params(
            rgb, output_size=(self.crop_size, self.crop_size)
        )
        rgb       = TF.crop(rgb,       i, j, h, w)
        normal    = TF.crop(normal,    i, j, h, w)
        roughness = TF.crop(roughness, i, j, h, w)
        metallic  = TF.crop(metallic,  i, j, h, w)
 
        if self.augment:
            rgb, normal, roughness, metallic = apply_geometric_augmentation(
                rgb, normal, roughness, metallic
            )
            rgb = apply_photometric_augmentation(rgb)
 
        rgb = (rgb - self._MEAN) / self._STD
        return {"rgb": rgb, "normal": normal,
                "roughness": roughness, "metallic": metallic, "group": group}
 
 
# ─── Guaranteed-metal batch sampler ───────────────────────────────────────────
 
class MetalGuaranteedSampler(BatchSampler):
    """
    Produces batches with exactly `n_metal` indices from the 'metal' group
    and `batch_size - n_metal` indices from all other groups.
 
    The metal pool is smaller (~202 training samples) than the non-metal pool
    (~2556). To avoid exhausting the metal pool early while still seeing the
    full non-metal distribution, the metal pool is re-shuffled and recycled
    whenever it runs out. The non-metal pool is traversed once per epoch.
 
    This results in approximately 2556 // 6 = 426 batches per epoch, with
    every non-metal texture seen once and each metal texture seen ~4 times.
    The heavy upsampling is necessary given the 1:12.6 imbalance and is
    complementary to BCEWithLogitsLoss(pos_weight=8.0) on the Metallic head.
    """
    def __init__(self, dataset, n_metal=2, batch_size=8, seed=42):
        self.metal_idx     = [i for i, g in enumerate(dataset.df[COL_GROUP])
                              if g == "metal"]
        self.non_metal_idx = [i for i, g in enumerate(dataset.df[COL_GROUP])
                              if g != "metal"]
        self.n_metal     = n_metal
        self.n_non_metal = batch_size - n_metal
        self.batch_size  = batch_size
        self.rng         = np.random.default_rng(seed)
 
    def __iter__(self):
        non_metal = self.non_metal_idx.copy()
        self.rng.shuffle(non_metal)
 
        metal_pool = []
 
        def next_metal():
            nonlocal metal_pool
            if not metal_pool:
                metal_pool = self.metal_idx.copy()
                self.rng.shuffle(metal_pool)
            return metal_pool.pop()
 
        n_batches = len(non_metal) // self.n_non_metal
        for i in range(n_batches):
            m_batch  = [next_metal() for _ in range(self.n_metal)]
            nm_batch = non_metal[i * self.n_non_metal:(i + 1) * self.n_non_metal]
            batch    = m_batch + nm_batch
            idx      = list(range(len(batch)))
            self.rng.shuffle(idx)
            yield [batch[j] for j in idx]
 
    def __len__(self):
        return len(self.non_metal_idx) // self.n_non_metal
 
 
# ─── Split loading (with fallback regeneration) ───────────────────────────────
 
def load_or_build_split(relabeling_csv, split_csv, val_fraction=0.15, seed=SEED):
    """
    Loads the frozen stratified split from `split_csv`. If the file does not
    exist (e.g. first run before the CSV was added to the dataset), regenerates
    it deterministically with `seed` and saves it to /kaggle/working/ as a
    fallback. The Kaggle dataset version is authoritative; the fallback is only
    for convenience.
    """
    if split_csv.exists():
        df = pd.read_csv(split_csv)
        assert "split" in df.columns, "split CSV missing 'split' column"
        print(f"Split loaded from dataset: {split_csv}")
    else:
        print("Split CSV not in dataset — regenerating with SEED=42 (fallback).")
        df_labels = pd.read_csv(relabeling_csv)
        rng = np.random.default_rng(seed)
        splits = []
        for group, gdf in df_labels.groupby(COL_GROUP):
            idx = gdf.index.to_numpy().copy()
            rng.shuffle(idx)
            n_val = max(1, int(len(idx) * val_fraction))
            splits.append(gdf.loc[idx[:n_val]].assign(split="val"))
            splits.append(gdf.loc[idx[n_val:]].assign(split="train"))
        df = pd.concat(splits).sort_index().reset_index(drop=True)
        fallback = OUTPUT_DIR / "matforge_split_fallback.csv"
        df.to_csv(fallback, index=False)
        print(f"Fallback split saved to {fallback}")
 
    n_tr = (df["split"] == "train").sum()
    n_va = (df["split"] == "val").sum()
    print(f"  Train: {n_tr}  Val: {n_va}  Total: {len(df)}")
    return df
 
 
def build_loaders(df_split, crop_size=256):
    """
    Builds train and validation DataLoaders.
 
    Training uses MetalGuaranteedSampler; validation uses sequential order.
    Returns (dl_train, dl_val, ds_train, ds_val).
    """
    ds_train = MatForgeDataset(df_split, "train", crop_size=crop_size, augment=True)
    ds_val   = MatForgeDataset(df_split, "val",   crop_size=crop_size, augment=False)
 
    sampler  = MetalGuaranteedSampler(ds_train, n_metal=N_METAL_PER_BATCH,
                                       batch_size=BATCH_SIZE, seed=SEED)
    dl_train = DataLoader(ds_train, batch_sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
    dl_val   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
 
    print(f"Train: {len(ds_train)} samples, {len(dl_train)} batches/epoch "
          f"(crop={crop_size})")
    print(f"Val  : {len(ds_val)} samples, {len(dl_val)} batches")
    return dl_train, dl_val, ds_train, ds_val
 
 
df_split = load_or_build_split(RELABELING_CSV, SPLIT_CSV)
dl_train, dl_val, ds_train, ds_val = build_loaders(df_split, crop_size=CROP_PHASE1)

Split loaded from dataset: /kaggle/input/datasets/mjgut05/matforge-pbr-dataset/matforge_split.csv
  Train: 2762  Val: 483  Total: 3245
Train: 2762 samples, 426 batches/epoch (crop=256)
Val  : 483 samples, 61 batches


## Cell 4 — MatForgeNet model

In [4]:
class FPNDecoder(nn.Module):
    """
    Top-down Feature Pyramid Network decoder.
 
    Fuses four encoder feature maps (L1..L4) into a single 256-channel map
    at 1/4 input resolution (64×64 for 256px input).
 
    At each level: the top-down stream is upsampled 2× and concatenated with
    the projected skip connection, then merged by a 3×3 conv → BN → ReLU.
    Concatenation (rather than addition) preserves complementary information
    from both streams, at the cost of doubling the channel count before the
    merging conv.
 
    Input channel counts for PVT-v2-B1:
        L1: 64  (H/4)
        L2: 128 (H/8)
        L3: 320 (H/16)
        L4: 512 (H/32)
    """
    def __init__(self, in_channels=(64, 128, 320, 512), out_channels=256):
        super().__init__()
        # 1×1 lateral projections: each encoder level → out_channels
        self.proj = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1, bias=False) for c in in_channels
        ])
        # Merging convs: concatenated (2*out_channels) → out_channels
        self.merge = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(out_channels * 2, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )
            for _ in range(len(in_channels) - 1)
        ])
 
    def forward(self, features):
        # features: [L1, L2, L3, L4] from encoder (low → high level)
        p = [self.proj[i](f) for i, f in enumerate(features)]
 
        # Top-down pass: start from coarsest (L4, index 3)
        x = p[3]
        for i in range(2, -1, -1):
            x = F.interpolate(x, size=p[i].shape[-2:], mode="bilinear",
                              align_corners=False)
            x = self.merge[i](torch.cat([x, p[i]], dim=1))
 
        return x   # (B, out_channels, H/4, W/4)
 
 
class RefineHead(nn.Module):
    """
    Per-output refine head: two progressive upsample blocks from 64×64 to 256×256,
    followed by a final projection to the target number of channels.
 
    Each block: Upsample 2× → Conv3×3 (in→128ch) → BN → ReLU → Conv3×3 (128→64ch)
    → BN → ReLU. The progressive refinement allows the network to add detail
    at each spatial scale independently.
    """
    def __init__(self, in_channels=256, out_channels=3):
        super().__init__()
 
        def up_block(ic, mid=128, oc=64):
            return nn.Sequential(
                nn.Conv2d(ic,  mid, 3, padding=1, bias=False),
                nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
                nn.Conv2d(mid, oc,  3, padding=1, bias=False),
                nn.BatchNorm2d(oc),  nn.ReLU(inplace=True),
            )
 
        self.block1 = up_block(in_channels)   # 64×64 → 128×128
        self.block2 = up_block(64)             # 128×128 → 256×256
        self.out    = nn.Conv2d(64, out_channels, 1)
 
    def forward(self, x):
        x = self.block1(F.interpolate(x, scale_factor=2, mode="bilinear",
                                       align_corners=False))
        x = self.block2(F.interpolate(x, scale_factor=2, mode="bilinear",
                                       align_corners=False))
        return self.out(x)
 
 
class MatForgeNet(nn.Module):
    """
    MatForge: PVT-v2-B1 hierarchical encoder + FPN decoder + three independent heads.
 
    Output:
        normal    (B, 3, H, W)  Tanh then L2-normalised per pixel; range [-1,1]
        roughness (B, 1, H, W)  Sigmoid; range [0,1]
        metallic  (B, 1, H, W)  Raw logits; sigmoid applied at inference time only
 
    The three heads are independent (no shared parameters after the FPN) to avoid
    gradient interference between tasks of fundamentally different natures: Normal
    is a unit-constrained vector field; Roughness and Metallic are scalar maps with
    no geometric constraint.
 
    The Metallic head emits logits (no activation) because the training loss is
    BCEWithLogitsLoss, which applies sigmoid internally for numerical stability.
    Applying sigmoid twice (in the head and in the loss) would saturate gradients.
    """
    def __init__(self, fpn_channels=256):
        super().__init__()
        self.encoder = timm.create_model(
            "pvt_v2_b1", pretrained=True, features_only=True
        )
        enc_channels = self.encoder.feature_info.channels()   # [64,128,320,512]
 
        self.fpn           = FPNDecoder(enc_channels, fpn_channels)
        self.head_normal   = RefineHead(fpn_channels, 3)
        self.head_roughness = RefineHead(fpn_channels, 1)
        self.head_metallic  = RefineHead(fpn_channels, 1)
 
    def forward(self, x):
        features  = self.encoder(x)          # [L1,L2,L3,L4]
        fpn_feat  = self.fpn(features)        # (B, 256, H/4, W/4)
 
        normal_raw = self.head_normal(fpn_feat)
        normal     = F.normalize(torch.tanh(normal_raw), dim=1, eps=1e-6)
 
        roughness  = torch.sigmoid(self.head_roughness(fpn_feat))
        metallic   = self.head_metallic(fpn_feat)   # logits
 
        return {"normal": normal, "roughness": roughness, "metallic": metallic}
 
    def set_encoder_grad(self, requires_grad: bool):
        """Freezes or unfreezes all encoder parameters."""
        for p in self.encoder.parameters():
            p.requires_grad = requires_grad
 
 
# ─── Instantiate and profile ──────────────────────────────────────────────────
model = MatForgeNet().to(DEVICE)
 
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
enc_params      = sum(p.numel() for p in model.encoder.parameters())
dec_params      = total_params - enc_params
 
print(f"Encoder params : {enc_params:>12,}")
print(f"Decoder params : {dec_params:>12,}")
print(f"Total params   : {total_params:>12,}")
print(f"Trainable      : {trainable_params:>12,}")
 
# Dry VRAM check
dummy = torch.randn(BATCH_SIZE, 3, CROP_PHASE1, CROP_PHASE1, device=DEVICE)
with torch.no_grad(), autocast():
    _ = model(dummy)
del dummy
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM after model forward (no grad):")
    print(f"  Allocated : {allocated:.2f} GB")
    print(f"  Reserved  : {reserved:.2f} GB")
    print(f"  Total     : {total_vram:.2f} GB")
    print(f"  Headroom  : {total_vram - reserved:.2f} GB")
torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/56.1M [00:00<?, ?B/s]

Encoder params :   13,496,000
Decoder params :    5,353,541
Total params   :   18,849,541
Trainable      :   18,849,541


/tmp/ipykernel_23/2353477746.py:145: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():



VRAM after model forward (no grad):
  Allocated : 0.09 GB
  Reserved  : 1.20 GB
  Total     : 15.64 GB
  Headroom  : 14.44 GB


## Cell 5 — Cook-Torrance differentiable renderer

In [5]:
def safe_normalize(x, dim=1, eps=1e-6):
    return x / torch.clamp(torch.linalg.norm(x, dim=dim, keepdim=True), min=eps)
 
 
def sample_lights(batch_size, num_lights, device, dtype,
                  cos_min=0.25, cos_max=0.92):
    """
    Samples random light directions in the upper hemisphere, avoiding extreme
    grazing angles (cos near 0 → black render, no gradient signal) and
    near-perpendicular lights (cos near 1 → saturated specular peak).
    """
    az  = 2.0 * math.pi * torch.rand(batch_size, num_lights, device=device, dtype=dtype)
    cos = cos_min + (cos_max - cos_min) * torch.rand(batch_size, num_lights, device=device, dtype=dtype)
    sin = torch.sqrt(torch.clamp(1.0 - cos ** 2, min=1e-6))
    dirs     = safe_normalize(torch.stack([torch.cos(az)*sin, torch.sin(az)*sin, cos], dim=-1), dim=-1)
    radiance = (0.9 + 0.2 * torch.rand(batch_size, num_lights, 1, device=device, dtype=dtype)).expand(-1, -1, 3)
    return dirs, radiance
 
 
def cook_torrance_render(albedo, N_pred, R_pred, M_pred,
                         light_dirs, light_rgb,
                         eps_dot=1e-4, eps_norm=1e-6, eps_denom=1e-6):
    """
    Differentiable flat-surface Cook-Torrance renderer (UE4 metallic/roughness).
 
    Implements: L_out = Σ_k [(k_D·albedo/π + f_s) · L_k · max(N·L_k, 0)]
    where f_s = D·G·F / (4·NoL·NoV).
 
    Components:
        D: GGX/Trowbridge-Reitz NDF with alpha = roughness^4
        G: Smith-Schlick geometry term (analytic-light variant)
        F: Schlick Fresnel; F0 = lerp(0.04, albedo, metallic)
 
    The loop over K lights is kept explicit (not vectorised) to avoid an
    (B,K,3,H,W) allocation that would OOM on 16 GB VRAM at batch=8.
 
    All denominators are clamped to prevent NaN/Inf gradients. See the four
    critical points documented in the architecture document (R3 risk).
    """
    B, _, H, W = albedo.shape
    K = light_dirs.shape[1]
 
    N     = safe_normalize(N_pred, dim=1, eps=eps_norm)
    rough = R_pred.clamp(0.0, 1.0)
    metal = M_pred.clamp(0.0, 1.0)
 
    # Orthographic view direction: valid for flat tileable textures.
    V = torch.zeros(B, 3, H, W, device=albedo.device, dtype=albedo.dtype)
    V[:, 2] = 1.0
 
    alpha  = rough * rough          # perceptual roughness → linear alpha
    alpha2 = alpha * alpha
    k_g    = (rough + 1.0) ** 2 / 8.0
    F0     = 0.04 * (1.0 - metal) + albedo * metal
 
    out = torch.zeros_like(albedo)
    for k_idx in range(K):
        L = safe_normalize(
            light_dirs[:, k_idx].view(B, 3, 1, 1).expand(B, 3, H, W),
            dim=1, eps=eps_norm,
        )
        H_vec = safe_normalize(L + V, dim=1, eps=eps_norm)
 
        NoL = (N * L).sum(1, keepdim=True).clamp(min=eps_dot, max=1.0)
        NoV = (N * V).sum(1, keepdim=True).clamp(min=eps_dot, max=1.0)
        NoH = (N * H_vec).sum(1, keepdim=True).clamp(min=eps_dot, max=1.0)
        VoH = (V * H_vec).sum(1, keepdim=True).clamp(min=eps_dot, max=1.0)
 
        # GGX NDF — critical point 3: clamp denominator
        denom = (NoH * NoH) * (alpha2 - 1.0) + 1.0
        D = alpha2 / torch.clamp(math.pi * denom * denom, min=eps_denom)
 
        # Smith-Schlick G
        Gv = NoV / torch.clamp(NoV * (1.0 - k_g) + k_g, min=1e-4)
        Gl = NoL / torch.clamp(NoL * (1.0 - k_g) + k_g, min=1e-4)
        G  = Gv * Gl
 
        # Schlick Fresnel
        Fspec = F0 + (1.0 - F0) * torch.pow(torch.clamp(1.0 - VoH, min=0.0), 5.0)
 
        # Critical point 4: 4·NoL·NoV denominator
        spec = (D * G / torch.clamp(4.0 * NoL * NoV, min=1e-4)) * Fspec
        kD   = (1.0 - Fspec) * (1.0 - metal)
        diff = kD * albedo / math.pi
 
        radiance = light_rgb[:, k_idx].view(B, 3, 1, 1)
        out = out + (diff + spec) * radiance * NoL
 
    return out / float(K)

## Cell 6 — Loss functions

In [6]:
def charbonnier_loss(pred, target, eps=0.001):
    """
    Charbonnier (pseudo-Huber) loss: sqrt((pred-target)^2 + eps^2).
    Behaves like L1 for large residuals and like L2 near zero, avoiding
    the non-differentiability of pure L1 at zero. eps=0.001 is appropriate
    for inputs in [-1,1] and [0,1].
    """
    return torch.sqrt((pred - target) ** 2 + eps ** 2).mean()
 
 
def cosine_normal_loss(pred, target, eps=1e-6):
    """
    Cosine distance between unit normal vectors.
    Returns 1 - mean(dot(N_pred_norm, N_gt_norm)); range [0, 2].
    L2 normalisation inside the loss provides a second safety renorm after the
    model's F.normalize, guarding against numerical drift in long runs.
    """
    pred_n = F.normalize(pred,   dim=1, eps=eps)
    tgt_n  = F.normalize(target, dim=1, eps=eps)
    return 1.0 - (pred_n * tgt_n).sum(dim=1).mean()
 
 
def sobel_gradients(x):
    """
    Computes Sobel gradients along X and Y using grouped depthwise convolution.
    Reflection padding avoids border artefacts.
    """
    C = x.shape[1]
    kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=x.dtype, device=x.device) / 8.0
    ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=x.dtype, device=x.device) / 8.0
    kx = kx.view(1, 1, 3, 3).expand(C, 1, 3, 3)
    ky = ky.view(1, 1, 3, 3).expand(C, 1, 3, 3)
    xp = F.pad(x, (1,1,1,1), mode="reflect")
    return F.conv2d(xp, kx, groups=C), F.conv2d(xp, ky, groups=C)
 
 
def gradient_loss(pred_n, gt_n, pred_r, gt_r):
    """
    Multi-scale Sobel gradient loss on Normal and Roughness maps.
    Applied at two scales (original and 2× downsampled) to capture
    both fine and coarse gradient structure. The gradient loss penalises
    discrepancies in edge sharpness and directional transitions without
    penalising overall magnitude, complementing Charbonnier.
    """
    total = 0.0
    for scale in [1, 2]:
        pn, gn = (pred_n, gt_n) if scale == 1 else (
            F.avg_pool2d(pred_n, scale), F.avg_pool2d(gt_n, scale))
        pr, gr = (pred_r, gt_r) if scale == 1 else (
            F.avg_pool2d(pred_r, scale), F.avg_pool2d(gt_r, scale))
 
        pn_gx, pn_gy = sobel_gradients(pn)
        gn_gx, gn_gy = sobel_gradients(gn)
        pr_gx, pr_gy = sobel_gradients(pr)
        gr_gx, gr_gy = sobel_gradients(gr)
 
        total += (torch.abs(pn_gx - gn_gx) + torch.abs(pn_gy - gn_gy)).mean()
        total += (torch.abs(pr_gx - gr_gx) + torch.abs(pr_gy - gr_gy)).mean()
 
    return total / (2 * 2)   # 2 maps × 2 scales
 
 
# BCEWithLogitsLoss for Metallic — pos_weight created once on the correct device
_metallic_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([METALLIC_POS_WEIGHT], device=DEVICE)
)
 
 
def compute_total_loss(outputs, targets, albedo, w_l1, w_lpips, lpips_fn):
    """
    Computes the full MatForge loss according to the schedule in the architecture
    document (v1.4, section 3):
 
        L = 1.0·L_normal + 0.8·L_roughness + 0.5·L_metallic
          + 0.15·L_grad + w_l1·L_render_L1 + w_lpips·L_render_LPIPS
 
    L_render weights are passed in from the progressive activation schedule;
    they are 0 during epochs 0-4 to prevent the renderer from dominating
    before the model has learned basic angular orientation.
 
    Returns a dict of individual loss values (detached floats) plus 'total'.
    """
    n_pred = outputs["normal"]
    r_pred = outputs["roughness"]
    m_logits = outputs["metallic"]
 
    n_gt = targets["normal"]
    r_gt = targets["roughness"]
    m_gt = targets["metallic"]
 
    l_normal    = cosine_normal_loss(n_pred, n_gt) + 0.25 * charbonnier_loss(n_pred, n_gt)
    l_roughness = charbonnier_loss(r_pred, r_gt)
    l_metallic  = _metallic_criterion(m_logits, m_gt)
    l_grad      = gradient_loss(n_pred, n_gt, r_pred, r_gt)
 
    l_render_l1   = torch.tensor(0.0, device=DEVICE)
    l_render_lpips = torch.tensor(0.0, device=DEVICE)
 
    if w_l1 > 0 or w_lpips > 0:
        B = albedo.shape[0]
        dirs, radiance = sample_lights(B, 3, DEVICE, albedo.dtype)
        m_prob = torch.sigmoid(m_logits)
        r_pred  = r_pred
 
        render_pred = cook_torrance_render(albedo, n_pred, r_pred, m_prob, dirs, radiance)
        render_gt   = cook_torrance_render(albedo, n_gt,   r_gt,  m_gt,  dirs, radiance)
 
        if w_l1 > 0:
            l_render_l1 = F.l1_loss(render_pred, render_gt)
 
        if w_lpips > 0:
            # LPIPS requires [-1,1] range; Reinhard tone-maps HDR render to [0,1]
            # then remapped to [-1,1]. Computed in float32 for stability.
            with torch.cuda.amp.autocast(enabled=False):
                def to_lpips(r):
                    r_f = r.float()
                    r_tm = r_f / (1.0 + r_f)     # Reinhard → [0,1]
                    return (r_tm * 2.0 - 1.0).clamp(-1.0, 1.0)
                l_render_lpips = lpips_fn(
                    to_lpips(render_pred), to_lpips(render_gt)
                ).mean()
 
    total = (W_NORMAL    * l_normal
           + W_ROUGHNESS * l_roughness
           + W_METALLIC  * l_metallic
           + W_GRAD      * l_grad
           + w_l1        * l_render_l1
           + w_lpips      * l_render_lpips)
 
    return {
        "total":    total,
        "normal":   l_normal.detach().item(),
        "roughness": l_roughness.detach().item(),
        "metallic": l_metallic.detach().item(),
        "grad":     l_grad.detach().item(),
        "render":   (w_l1 * l_render_l1 + w_lpips * l_render_lpips).detach().item(),
    }

## Cell 7 — Metrics and LPIPS evaluation model

In [7]:
def mean_angular_error_deg(pred, target, eps_norm=1e-6, eps_acos=1e-7):
    """Mean angular error in degrees between predicted and GT normal maps."""
    pred_n = F.normalize(pred,   dim=1, eps=eps_norm)
    tgt_n  = F.normalize(target, dim=1, eps=eps_norm)
    dot    = (pred_n * tgt_n).sum(dim=1).clamp(-1.0 + eps_acos, 1.0 - eps_acos)
    return (torch.acos(dot) * (180.0 / math.pi)).mean().item()
 
 
def metallic_recall(pred_logits, target, threshold=0.5):
    """
    Recall of the Metallic head on positive pixels (metallic GT > threshold).
    A near-zero recall in early epochs indicates head collapse to all-zero.
    """
    pred_bin = (torch.sigmoid(pred_logits) > threshold).float()
    pos_mask = (target > threshold)
    if pos_mask.sum() == 0:
        return float("nan")
    return (pred_bin[pos_mask] == 1.0).float().mean().item()
 
 
# LPIPS evaluation network (eval-only, no gradients, fixed weights)
lpips_eval_fn = lpips_lib.LPIPS(net="alex").to(DEVICE).eval()
for p in lpips_eval_fn.parameters():
    p.requires_grad = False
 
# Training LPIPS (same network, shared weights — only used when w_lpips > 0)
lpips_train_fn = lpips_eval_fn   # same instance; always in eval mode
 
 
def compute_render_lpips_metric(model_out, targets, albedo):
    """
    Computes LPIPS between rendered images for validation metrics.
    Uses 5 fixed light directions (more than training for stable estimates).
    """
    with torch.no_grad():
        torch.manual_seed(0)   # fixed lights for reproducible validation metric
        dirs, radiance = sample_lights(albedo.shape[0], 5, DEVICE, albedo.dtype)
        m_prob = torch.sigmoid(model_out["metallic"])
        rp = cook_torrance_render(albedo, model_out["normal"],
                                  model_out["roughness"], m_prob, dirs, radiance)
        rg = cook_torrance_render(albedo, targets["normal"],
                                  targets["roughness"], targets["metallic"], dirs, radiance)
        def to_lpips(r):
            r_tm = r / (1.0 + r)
            return (r_tm * 2.0 - 1.0).clamp(-1.0, 1.0)
        return lpips_eval_fn(to_lpips(rp), to_lpips(rg)).mean().item()

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 197MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


## Cell 8 — Optimizer, scheduler, and EMA

In [8]:
class EMA:
    """
    Exponential Moving Average of model parameters.
 
    Maintains a shadow copy of all trainable parameters updated as:
        shadow = decay * shadow + (1 - decay) * param
 
    `apply_shadow()` / `restore()` are called around validation to use
    the smoothed weights without modifying the training weights permanently.
 
    EMA is activated only after the encoder unfreezes (epoch FREEZE_EPOCHS),
    because shadow-averaging frozen parameters provides no benefit and
    wastes memory bandwidth.
    """
    def __init__(self, model, decay=0.999):
        self.model  = model
        self.decay  = decay
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters()
                       if p.requires_grad}
        self.backup = {}
 
    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name] = (self.decay * self.shadow[name]
                                     + (1.0 - self.decay) * param.data)
 
    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]
 
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name]
        self.backup.clear()
 
 
# ─── Freeze encoder for first FREEZE_EPOCHS epochs ───────────────────────────
model.set_encoder_grad(False)
 
# ─── Optimizer: separate LR groups for encoder and decoder+heads ─────────────
optimizer = AdamW([
    {"params": model.encoder.parameters(),    "lr": LR_ENCODER},
    {"params": list(model.fpn.parameters()) +
               list(model.head_normal.parameters()) +
               list(model.head_roughness.parameters()) +
               list(model.head_metallic.parameters()), "lr": LR_DECODER},
], weight_decay=WEIGHT_DECAY)
 
# ─── Scheduler: linear warmup → cosine decay ─────────────────────────────────
n_cosine     = max(1, TOTAL_EPOCHS - WARMUP_EPOCHS)
warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                         total_iters=WARMUP_EPOCHS)
cosine_sched = CosineAnnealingLR(optimizer, T_max=n_cosine, eta_min=1e-6)
scheduler    = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched],
                             milestones=[WARMUP_EPOCHS])

scaler = GradScaler()
ema    = None

print("Optimizer, scheduler, and GradScaler ready.")
print(f"Encoder LR    : {LR_ENCODER}  (frozen for first {FREEZE_EPOCHS} epochs)")
print(f"Decoder LR    : {LR_DECODER}")
print(f"Total epochs  : {TOTAL_EPOCHS}")

Optimizer, scheduler, and GradScaler ready.
Encoder LR    : 0.0001  (frozen for first 3 epochs)
Decoder LR    : 0.0003
Total epochs  : 90


/tmp/ipykernel_23/1126005464.py:61: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


## Extra Cell — Multi-scale conditional PatchGAN discriminator

In [9]:
def _disc_block(in_ch, out_ch, norm=True):
    """Single conv block for the discriminator."""
    layers = [nn.utils.spectral_norm(
        nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=False)
    )]
    if norm:
        layers.append(nn.InstanceNorm2d(out_ch, affine=False))
    layers.append(nn.LeakyReLU(0.2, inplace=True))
    return layers


class PatchDiscriminator(nn.Module):
    """
    70×70 conditional PatchGAN discriminator with spectral normalization.

    Input: (B, 8, H, W) — concatenation of RGB(3) + Normal(3) + Roughness(1) + Metallic(1).
    Output: (B, 1, H/16, W/16) — real/fake score map per patch.

    Spectral normalization on every Conv2d constrains the Lipschitz constant of
    the discriminator without requiring gradient penalty hyperparameter tuning.
    InstanceNorm is preferred over BatchNorm because batch statistics mix across
    texture groups with different mean roughness/normal distributions.
    Dropout (p=0.1) provides mild regularisation against discriminator overfitting
    on the 3,245-texture dataset.

    Feature maps from intermediate layers are returned alongside the logit map
    to enable the feature matching loss.
    """
    IN_CH = 8   # RGB(3) + Normal(3) + Roughness(1) + Metallic(1)

    def __init__(self, dropout_p: float = 0.1):
        super().__init__()
        self.block0 = nn.Sequential(*_disc_block(self.IN_CH, 64, norm=False))
        self.drop0  = nn.Dropout2d(dropout_p)
        self.block1 = nn.Sequential(*_disc_block(64,  128))
        self.block2 = nn.Sequential(*_disc_block(128, 256))
        self.block3 = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1, bias=False)
            ),
            nn.InstanceNorm2d(512, affine=False),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.out = nn.utils.spectral_norm(
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)
        )

    def forward(self, x):
        """Returns (logit_map, [feat0, feat1, feat2, feat3]) for feature matching."""
        f0 = self.drop0(self.block0(x))
        f1 = self.block1(f0)
        f2 = self.block2(f1)
        f3 = self.block3(f2)
        logit = self.out(f3)
        return logit, [f0, f1, f2, f3]


def build_discriminators(device):
    """
    Builds two PatchDiscriminators at different scales.
    D1 operates on the original resolution.
    D2 operates on a 2× downsampled version of the same input, giving a larger
    effective receptive field to penalise medium-frequency blurriness.
    """
    D1 = PatchDiscriminator().to(device)
    D2 = PatchDiscriminator().to(device)
    n_params = sum(p.numel() for p in D1.parameters()) * 2
    print(f"Discriminators: 2 × PatchDiscriminator")
    print(f"Total D params : {n_params:,}")
    return D1, D2


def lsgan_loss_D(logit_real, logit_fake):
    """LSGAN discriminator loss: MSE toward 1 for real, 0 for fake."""
    return 0.5 * (F.mse_loss(logit_real, torch.ones_like(logit_real))
                + F.mse_loss(logit_fake, torch.zeros_like(logit_fake)))


def lsgan_loss_G(logit_fake):
    """LSGAN generator adversarial loss: MSE toward 1 (fool discriminator)."""
    return F.mse_loss(logit_fake, torch.ones_like(logit_fake))


def feature_matching_loss(feats_real, feats_fake):
    """
    L1 distance between discriminator intermediate feature maps for real and
    fake inputs. Provides a stable gradient signal to the generator even when
    the discriminator is still learning, preventing early generator collapse.
    """
    loss = 0.0
    for fr, ff in zip(feats_real, feats_fake):
        loss += F.l1_loss(ff, fr.detach())
    return loss / len(feats_real)


def r1_gradient_penalty(D, real_input):
    """
    Lazy R1 penalty computed in float32 regardless of AMP context.

    torch.autograd.grad over autocast (float16) tensors produces NaN because
    the intermediate activations lose precision. Casting to float32 before the
    forward pass of D and the gradient computation resolves this without
    affecting the AMP training of the rest of the loop.
    """
    real_f32 = real_input.detach().float().requires_grad_(True)
    with torch.amp.autocast("cuda", enabled=False):
        logit_real, _ = D(real_f32)
        logit_real = logit_real.float()
    grad = torch.autograd.grad(
        outputs=logit_real.sum(), inputs=real_f32,
        create_graph=True, retain_graph=True, only_inputs=True,
    )[0]
    penalty = (grad.view(grad.shape[0], -1).norm(2, dim=1) ** 2).mean()
    return penalty * (R1_LAMBDA / 2) * R1_INTERVAL


def build_disc_input(rgb, normal, roughness, metallic, device):
    """
    Concatenates RGB + all PBR maps into the 8-channel discriminator input.
    All maps must be in their natural range: RGB [0,1], Normal [-1,1],
    Roughness [0,1], Metallic [0,1].
    """
    return torch.cat([rgb, normal, roughness, metallic], dim=1).to(device)


if GAN_ACTIVE:
    D1, D2 = build_discriminators(DEVICE)
    optimizer_D = torch.optim.AdamW(
        list(D1.parameters()) + list(D2.parameters()),
        lr=LR_D, betas=(0.0, 0.99), weight_decay=0.0
    )
    scaler_D = GradScaler()
    print("Discriminator optimizers ready.")

Discriminators: 2 × PatchDiscriminator
Total D params : 5,537,794
Discriminator optimizers ready.


/tmp/ipykernel_23/2152400245.py:132: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_D = GradScaler()


## Cell 9 — Checkpoint utilities

In [10]:
def save_checkpoint(path, model, optimizer, scheduler, scaler, ema,
                    epoch, metrics, best_metrics, full=True):
    """
    Saves a training checkpoint.
 
    `full=True`  → includes optimizer, scheduler, scaler, EMA (for resumption).
    `full=False` → model state dict only (periodic epoch snapshots, smaller files).
    """
    payload = {
        "epoch":        epoch,
        "model":        model.state_dict(),
        "metrics":      metrics,
        "best_metrics": best_metrics,
    }
    if full:
        payload.update({
            "optimizer":  optimizer.state_dict(),
            "scheduler":  scheduler.state_dict(),
            "scaler":     scaler.state_dict(),
            "ema_shadow": ema.shadow if ema is not None else None,
        })
    torch.save(payload, path)
 
 
def load_checkpoint(path, model, optimizer=None, scheduler=None,
                    scaler=None, device=DEVICE):
    """
    Loads a checkpoint and restores model (and optionally optimizer) state.
    Returns (epoch, metrics, best_metrics, ema_shadow_or_None).
    """
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    print(f"Checkpoint loaded from {path} (epoch {ckpt['epoch']})")
    return ckpt["epoch"], ckpt.get("metrics", {}), ckpt.get("best_metrics", {}), ckpt.get("ema_shadow")

## Cell 10 — Reference textures for validation panel

In [11]:
_MEAN_T = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD_T  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
 
 
def load_reference_texture(filename, crop_size=256):
    """
    Loads a single texture from disk and returns a dict of tensors ready for
    model input and display. Uses a deterministic centre crop so that the
    same region is shown across all epochs in the validation panel.
    """
    def load(path, mode):
        return TF.to_tensor(Image.open(path).convert(mode))
 
    rgb_raw   = load(RGB_DIR       / filename, "RGB")
    normal_01 = load(NORMAL_DIR    / filename, "RGB")
    roughness = load(ROUGHNESS_DIR / filename, "L")
 
    metal_path = METALLIC_DIR / filename
    if metal_path.exists():
        metallic = load(metal_path, "L")
    else:
        metallic = torch.zeros(1, rgb_raw.shape[1], rgb_raw.shape[2])
 
    normal = normal_01 * 2.0 - 1.0
 
    # Centre crop for consistent framing across epochs
    h, w    = rgb_raw.shape[1], rgb_raw.shape[2]
    i       = max(0, (h - crop_size) // 2)
    j       = max(0, (w - crop_size) // 2)
    cs      = crop_size
 
    def cc(t): return TF.crop(t, i, j, cs, cs)
 
    rgb_crop = cc(rgb_raw)
    return {
        "filename":    filename,
        "rgb_display": rgb_crop,                          # [0,1], for display
        "rgb_input":   (rgb_crop - _MEAN_T) / _STD_T,    # ImageNet-normed, for model
        "normal_gt":   cc(normal),
        "roughness_gt": cc(roughness),
        "metallic_gt": cc(metallic),
    }

# Hardcoded selection — one texture per group, manually verified
_GROUP_ILOC = {
    "brick_terracotta": 7,
    "ceramic_ground":   0,
    "concrete_plaster": 2,
    "marble_smooth":    9,
    "metal":            None,   # direct filename below
    "mixed_ambiguous":  8,
    "stone_rough":      0,
    "wood":             26,
}
_METAL_REF = "metal_0081.png"

def build_reference_set(df_split, crop_size=256):
    refs = []
    for group in GROUPS:
        iloc_idx = _GROUP_ILOC[group]
        if iloc_idx is None:
            fname = _METAL_REF
        else:
            gdf = df_split[(df_split[COL_GROUP] == group) &
                           (df_split["split"] == "val")]
            if len(gdf) == 0:
                gdf = df_split[df_split[COL_GROUP] == group]
            fname = gdf.sort_values(COL_FILENAME).iloc[iloc_idx][COL_FILENAME]
        tex = load_reference_texture(fname, crop_size)
        tex["label"] = group
        refs.append(tex)
    print(f"Reference set: {len(refs)} textures (hardcoded per group)")
    return refs

reference_set = build_reference_set(df_split)

Reference set: 8 textures (hardcoded per group)


## Cell 11 — Validation function

In [12]:
@torch.no_grad()
def validate(model, dl_val, lpips_fn, epoch, ema=None, generate_panel=True):
    """
    Runs full validation on the val DataLoader and generates a visual panel.
 
    If `ema` is provided, applies EMA shadow weights for the forward passes
    and restores the original weights before returning.
 
    Returns a metrics dict:
        mae_normal_deg, cos_dist, mae_roughness, rmse_roughness,
        lpips_render, metallic_recall, S
    """
    if ema is not None:
        ema.apply_shadow()
 
    model.eval()
 
    acc = defaultdict(float)
    n_batches = 0
 
    for batch in dl_val:
        rgb       = batch["rgb"].to(DEVICE)
        normal_gt = batch["normal"].to(DEVICE)
        rough_gt  = batch["roughness"].to(DEVICE)
        metal_gt  = batch["metallic"].to(DEVICE)
        albedo_raw = (rgb * _STD_T.to(DEVICE)) + _MEAN_T.to(DEVICE)  # un-normalise for renderer
 
        with autocast():
            out = model(rgb)
 
        acc["mae_normal"]  += mean_angular_error_deg(out["normal"], normal_gt)
        acc["cos_dist"]    += (1.0 - (F.normalize(out["normal"], dim=1) *
                                       F.normalize(normal_gt, dim=1)).sum(1).mean().item())
        diff_r = (out["roughness"] - rough_gt).abs()
        acc["mae_roughness"]  += diff_r.mean().item()
        acc["rmse_roughness"] += (diff_r ** 2).mean().sqrt().item()
 
        targets = {"normal": normal_gt, "roughness": rough_gt, "metallic": metal_gt}
        acc["lpips_render"] += compute_render_lpips_metric(out, targets, albedo_raw)
 
        # Metallic recall only over metal-group samples
        metal_mask = (metal_gt.amax(dim=(1,2,3)) > 0.05)
        if metal_mask.any():
            acc["metallic_recall"] += metallic_recall(
                out["metallic"][metal_mask], metal_gt[metal_mask]
            )
            acc["metallic_batches"] += 1.0
 
        n_batches += 1
 
    metrics = {k: v / n_batches for k, v in acc.items()
               if k != "metallic_batches"}
    if acc["metallic_batches"] > 0:
        metrics["metallic_recall"] = acc["metallic_recall"] / acc["metallic_batches"]
    else:
        metrics["metallic_recall"] = float("nan")
 
    # Composite stopping metric S (lower is better)
    lpips_val = metrics.get("lpips_render", 1.0)
    metrics["S"] = (metrics["mae_normal"] + 0.6 * metrics["mae_roughness"]
                    + 0.2 * lpips_val)
 
    # ─── Visual validation panel ──────────────────────────────────────────────
    if generate_panel:
        n_refs = len(reference_set)
        fig, axes = plt.subplots(n_refs, 7, figsize=(16, 2.2 * n_refs))
        fig.suptitle(f"Validation panel — Epoch {epoch:03d}", fontsize=10, y=1.01)
        col_titles = ["RGB in", "Normal GT", "Normal pred",
                      "Rough GT", "Rough pred", "Metal GT", "Metal pred"]
        for ci, ct in enumerate(col_titles):
            axes[0][ci].set_title(ct, fontsize=7)
 
        for ri, ref in enumerate(reference_set):
            rgb_in  = ref["rgb_input"].unsqueeze(0).to(DEVICE)
            with autocast():
                pred = model(rgb_in)
 
            def show(ax, t, cmap=None, vmin=0, vmax=1):
                if t.ndim == 3 and t.shape[0] == 3:
                    img = t.permute(1, 2, 0).cpu().float().numpy()
                else:
                    img = t.squeeze().cpu().float().numpy()
                ax.imshow(img.clip(0, 1) if cmap is None else img,
                          cmap=cmap, vmin=vmin, vmax=vmax)
                ax.axis("off")
 
            def unorm(t):
                return ((t * _STD_T) + _MEAN_T).clamp(0, 1)
 
            def norm_to_img(t):
                return ((t + 1.0) / 2.0).clamp(0, 1)
 
            show(axes[ri][0], unorm(ref["rgb_input"]))
            show(axes[ri][1], norm_to_img(ref["normal_gt"]))
            show(axes[ri][2], norm_to_img(pred["normal"].squeeze(0).detach()))
            show(axes[ri][3], ref["roughness_gt"], cmap="gray")
            show(axes[ri][4], pred["roughness"].squeeze().detach(), cmap="gray")
            show(axes[ri][5], ref["metallic_gt"], cmap="gray")
            show(axes[ri][6], torch.sigmoid(pred["metallic"]).squeeze().detach(), cmap="gray")
 
            axes[ri][0].set_ylabel(ref["label"].replace("_", "\n"),
                       fontsize=7, fontweight="bold", rotation=0,
                       labelpad=72, va="center")
 
        plt.tight_layout()
        panel_path = PANELS_DIR / f"panel_ep{epoch:03d}.png"
        plt.savefig(panel_path, dpi=90, bbox_inches="tight")
        plt.close(fig)
 
    if ema is not None:
        ema.restore()
 
    model.train()
    return metrics

## Cell 12 — Training loop

In [13]:
# ─── Resume configuration ─────────────────────────────────────────────────────
RESUME_FROM = None  # supervised training already complete, skip to GAN

start_epoch  = TOTAL_EPOCHS  # supervised loop complete — skip to GAN phase
best_metrics = {"S": float("inf"), "mae_normal": float("inf"),
                "mae_roughness": float("inf"), "epoch": -1}
ema_active   = False

if RESUME_FROM and Path(RESUME_FROM).exists():
    # Load model and optimizer states; scheduler is NOT restored because
    # the previous cosine cycle completed at epoch 19 (LR=1e-6). Restoring
    # it would keep LR at minimum for all remaining epochs. Instead, a fresh
    # cosine restart is built below covering epochs 20-89.
    start_epoch, _, best_metrics, ema_shadow = load_checkpoint(
        RESUME_FROM, model, optimizer, scheduler=None, scaler=scaler
    )
    start_epoch += 1

    # Encoder must be unfrozen before EMA is created so that encoder params
    # are included in the shadow dict.
    model.set_encoder_grad(True)

    if ema_shadow is not None:
        ema = EMA(model, decay=EMA_DECAY)
        ema.shadow = ema_shadow
        ema_active = True

    # Cosine restart for epochs 20-89: peak LR reduced to 30% of original
    # to avoid disturbing converged weights. No warmup needed.
    remaining = TOTAL_EPOCHS - start_epoch
    optimizer.param_groups[0]["lr"] = LR_ENCODER_RESUME
    optimizer.param_groups[1]["lr"] = LR_DECODER_RESUME
    scheduler = CosineAnnealingLR(
        optimizer, T_max=max(1, remaining), eta_min=1e-6
    )
    print(f"Resumed from epoch {start_epoch - 1}. "
          f"Cosine restart: LR enc={LR_ENCODER_RESUME:.1e} "
          f"dec={LR_DECODER_RESUME:.1e} over {remaining} epochs.")

# ─── Training loop ────────────────────────────────────────────────────────────
print("Supervised training complete. Skipping to GAN phase.")
print(f"\nStarting training: epoch {start_epoch} → {TOTAL_EPOCHS - 1}")
print("=" * 72)
 
for epoch in range(start_epoch, TOTAL_EPOCHS):
    epoch_start = time.perf_counter()
 
    # ── Encoder freeze / unfreeze ─────────────────────────────────────────────
    if epoch == FREEZE_EPOCHS:
        model.set_encoder_grad(True)
        ema = EMA(model, decay=EMA_DECAY)
        ema_active = True
        print(f"[Epoch {epoch:03d}] Encoder unfrozen. EMA activated.")
 
    # ── Curriculum resolution ─────────────────────────────────────────────────
    if not DRY_RUN and epoch == CURRICULUM_EPOCH:
        dl_train, dl_val, ds_train, ds_val = build_loaders(
            df_split, crop_size=CROP_PHASE2
        )
        reference_set = build_reference_set(df_split, crop_size=CROP_PHASE2)
        print(f"[Epoch {epoch:03d}] Curriculum: crop size → {CROP_PHASE2}px")
 
    # ── Render loss weights for this epoch ────────────────────────────────────
    w_l1, w_lpips = get_render_weights(epoch)
 
    # ── Training pass ─────────────────────────────────────────────────────────
    model.train()
    loss_accum = defaultdict(float)
    n_steps = 0
 
    for batch in dl_train:
        rgb       = batch["rgb"].to(DEVICE,       non_blocking=True)
        normal_gt = batch["normal"].to(DEVICE,    non_blocking=True)
        rough_gt  = batch["roughness"].to(DEVICE, non_blocking=True)
        metal_gt  = batch["metallic"].to(DEVICE,  non_blocking=True)
 
        # Un-normalise albedo for the renderer (renderer expects linear [0,1])
        albedo = (rgb * _STD_T.to(DEVICE)) + _MEAN_T.to(DEVICE)
 
        optimizer.zero_grad(set_to_none=True)
 
        with autocast():
            outputs = model(rgb)
            targets = {"normal": normal_gt, "roughness": rough_gt, "metallic": metal_gt}
            loss_dict = compute_total_loss(
                outputs, targets, albedo.detach(),
                w_l1, w_lpips, lpips_train_fn
            )
 
        scaler.scale(loss_dict["total"]).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
 
        if ema_active:
            ema.update()
 
        for k, v in loss_dict.items():
            if k != "total":
                loss_accum[k] += v if isinstance(v, float) else v.item()
        loss_accum["total"] += loss_dict["total"].detach().item()
        n_steps += 1
 
    scheduler.step()
    epoch_loss = {k: v / n_steps for k, v in loss_accum.items()}
    epoch_time = time.perf_counter() - epoch_start
 
    # ── Validation ────────────────────────────────────────────────────────────
    do_validate = (epoch % VALIDATE_EVERY == 0) or (epoch == TOTAL_EPOCHS - 1)
    do_panel    = do_validate   # always generate panel alongside metrics
 
    if do_validate:
        val_metrics = validate(
            model, dl_val, lpips_eval_fn, epoch,
            ema=ema if ema_active else None,
            generate_panel=do_panel,
        )
 
        # ── Checkpoint: save best models ──────────────────────────────────────
        saved = []
        if val_metrics["S"] < best_metrics["S"]:
            best_metrics.update(val_metrics)
            best_metrics["epoch"] = epoch
            save_checkpoint(CKPT_DIR / "best_overall.pt", model, optimizer,
                            scheduler, scaler, ema, epoch, val_metrics, best_metrics)
            saved.append("best_overall.pt")
        if val_metrics["mae_normal"] < best_metrics.get("mae_normal_best", float("inf")):
            best_metrics["mae_normal_best"] = val_metrics["mae_normal"]
            save_checkpoint(CKPT_DIR / "best_normal.pt", model, None,
                            None, None, ema, epoch, val_metrics, best_metrics, full=False)
            saved.append("best_normal.pt")
        if val_metrics["mae_roughness"] < best_metrics.get("mae_rough_best", float("inf")):
            best_metrics["mae_rough_best"] = val_metrics["mae_roughness"]
            save_checkpoint(CKPT_DIR / "best_roughness.pt", model, None,
                            None, None, ema, epoch, val_metrics, best_metrics, full=False)
            saved.append("best_roughness.pt")
 
        if epoch % SAVE_CKPT_EVERY == 0:
            ckpt_name = f"checkpoint_ep{epoch:03d}.pt"
            save_checkpoint(CKPT_DIR / ckpt_name, model, None,
                            None, None, ema, epoch, val_metrics, best_metrics, full=False)
            saved.append(ckpt_name)
 
        # Always overwrite last.pt with full state for resumption
        save_checkpoint(CKPT_DIR / "last.pt", model, optimizer,
                        scheduler, scaler, ema, epoch, val_metrics, best_metrics)
        saved.append("last.pt")
 
        # ── Current LR ────────────────────────────────────────────────────────
        lr_enc = optimizer.param_groups[0]["lr"]
        lr_dec = optimizer.param_groups[1]["lr"]
        enc_status = "FROZEN" if epoch < FREEZE_EPOCHS else "active"
        ema_status = "ON" if ema_active else "OFF"
        render_status = f"δ₁={w_l1:.2f} δ₂={w_lpips:.2f}"
 
        # ── Monitoring block ───────────────
        mins, secs = divmod(int(epoch_time), 60)
        print("=" * 72)
        print(f"EPOCH {epoch:03d} | {mins}m {secs:02d}s | "
              f"LR enc={lr_enc:.2e} dec={lr_dec:.2e}")
        print(f"ENCODER: {enc_status} | EMA: {ema_status} | "
              f"L_render: {render_status}")
        print("-" * 72)
        print("TRAIN LOSSES (epoch avg)")
        print(f"  total={epoch_loss['total']:.4f}  "
              f"normal={epoch_loss['normal']:.4f}  "
              f"rough={epoch_loss['roughness']:.4f}  "
              f"metal={epoch_loss['metallic']:.4f}  "
              f"grad={epoch_loss['grad']:.4f}  "
              f"render={epoch_loss['render']:.4f}")
        print("VALIDATION METRICS")
        lp = val_metrics.get("lpips_render", float("nan"))
        mr = val_metrics.get("metallic_recall", float("nan"))
        print(f"  Normal   : MAE={val_metrics['mae_normal']:.2f}°  "
              f"cos_dist={val_metrics['cos_dist']:.4f}")
        print(f"  Roughness: MAE={val_metrics['mae_roughness']:.4f}  "
              f"RMSE={val_metrics['rmse_roughness']:.4f}")
        print(f"  Render   : LPIPS={lp:.4f}")
        print(f"  Metallic : Recall={mr * 100:.1f}%")
        print(f"  S composite: {val_metrics['S']:.4f}  "
              f"(best: {best_metrics['S']:.4f} @ ep{best_metrics['epoch']:03d})")
        print(f"CHECKPOINTS: {', '.join(saved) if saved else 'none'}")
        print(f"PANEL: panels/panel_ep{epoch:03d}.png")
        print("=" * 72)

# ─── GAN fine-tuning phase ────────────────────────────────────────────────────
if GAN_ACTIVE:

    # Load best supervised checkpoint as starting point
    gan_ckpt_path = GAN_CKPT_SOURCE / "best_overall.pt"
    gan_disc_path = GAN_CKPT_SOURCE / "best_gan_disc.pt"

    if gan_ckpt_path.exists():
        ckpt = torch.load(gan_ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        if ckpt.get("ema_shadow") is not None:
            ema = EMA(model, decay=EMA_DECAY)
            ema.shadow = ckpt["ema_shadow"]
            ema_active = True
        print(f"GAN phase: loaded generator from best_overall.pt (epoch {ckpt['epoch']})")
    else:
        print("WARNING: best_overall.pt not found")

    if gan_disc_path.exists():
        disc_ckpt = torch.load(gan_disc_path, map_location=DEVICE)
        D1.load_state_dict(disc_ckpt["D1"])
        D2.load_state_dict(disc_ckpt["D2"])
        optimizer_D.load_state_dict(disc_ckpt["optimizer_D"])
        print(f"GAN phase: loaded discriminator from epoch_gan={disc_ckpt['epoch_gan']}")
    else:
        print("GAN phase: discriminator initialized from scratch")

    model.set_encoder_grad(True)

    # Separate LR groups maintained from supervised phase; reduce for GAN stability
    optimizer_G_gan = AdamW([
        {"params": model.encoder.parameters(),    "lr": LR_ENCODER_RESUME * 0.5},
        {"params": list(model.fpn.parameters()) +
                   list(model.head_normal.parameters()) +
                   list(model.head_roughness.parameters()) +
                   list(model.head_metallic.parameters()),
         "lr": LR_DECODER_RESUME * 0.5},
    ], weight_decay=WEIGHT_DECAY)
    scheduler_G_gan = CosineAnnealingLR(
        optimizer_G_gan, T_max=max(1, GAN_TOTAL), eta_min=1e-6
    )

    best_S_gan = best_metrics.get("S", float("inf"))
    step_counter = 0

    print(f"\nStarting GAN fine-tuning: {GAN_TOTAL} epochs")
    print("=" * 72)

    for epoch_gan in range(GAN_TOTAL):
        epoch_start = time.perf_counter()
        w_adv = get_adv_weight(epoch_gan)
        w_l1, w_lpips = get_render_weights(90 + epoch_gan)  # always full render

        model.train(); D1.train(); D2.train()

        acc_G = defaultdict(float)
        acc_D = defaultdict(float)
        n_steps = 0

        for batch in dl_train:
            rgb       = batch["rgb"].to(DEVICE,       non_blocking=True)
            normal_gt = batch["normal"].to(DEVICE,    non_blocking=True)
            rough_gt  = batch["roughness"].to(DEVICE, non_blocking=True)
            metal_gt  = batch["metallic"].to(DEVICE,  non_blocking=True)
            albedo    = (rgb * _STD_T.to(DEVICE)) + _MEAN_T.to(DEVICE)

            # Un-normalised RGB for discriminator input (visual range)
            rgb_vis = albedo.clamp(0, 1)

            # ── 1. Update discriminator ───────────────────────────────────────
            with torch.no_grad(), autocast():
                out_fake = model(rgb)

            m_fake = torch.sigmoid(out_fake["metallic"])

            real_in1 = build_disc_input(rgb_vis, normal_gt, rough_gt, metal_gt, DEVICE)
            fake_in1 = build_disc_input(rgb_vis,
                                         out_fake["normal"].detach(),
                                         out_fake["roughness"].detach(),
                                         m_fake.detach(), DEVICE)
            real_in2 = F.avg_pool2d(real_in1, 2)
            fake_in2 = F.avg_pool2d(fake_in1, 2)

            optimizer_D.zero_grad(set_to_none=True)
            with autocast():
                lr1, _ = D1(real_in1); lf1, _ = D1(fake_in1)
                lr2, _ = D2(real_in2); lf2, _ = D2(fake_in2)
                loss_D = (lsgan_loss_D(lr1, lf1) + lsgan_loss_D(lr2, lf2)) * 0.5

            scaler_D.scale(loss_D).backward()

            # Lazy R1 penalty — computed in float32, outside autocast
            if step_counter % R1_INTERVAL == 0:
                r1 = (r1_gradient_penalty(D1, real_in1)
                     + r1_gradient_penalty(D2, real_in2)) * 0.5
                scaler_D.scale(r1).backward()

            scaler_D.unscale_(optimizer_D)
            torch.nn.utils.clip_grad_norm_(
                list(D1.parameters()) + list(D2.parameters()), max_norm=10.0
            )
            scaler_D.step(optimizer_D)
            scaler_D.update()

            # ── 2. Update generator ───────────────────────────────────────────
            optimizer_G_gan.zero_grad(set_to_none=True)
            with autocast():
                out = model(rgb)
                targets = {"normal": normal_gt, "roughness": rough_gt,
                           "metallic": metal_gt}
                sup_losses = compute_total_loss(
                    out, targets, albedo.detach(),
                    w_l1, w_lpips, lpips_train_fn
                )

                m_prob = torch.sigmoid(out["metallic"])
                fake_in1_G = build_disc_input(rgb_vis, out["normal"],
                                               out["roughness"], m_prob, DEVICE)
                fake_in2_G = F.avg_pool2d(fake_in1_G, 2)

                lf1_G, feats_fake1 = D1(fake_in1_G)
                lf2_G, feats_fake2 = D2(fake_in2_G)
                with torch.no_grad():
                    _, feats_real1 = D1(real_in1.detach())
                    _, feats_real2 = D2(real_in2.detach())

                l_adv = (lsgan_loss_G(lf1_G) + lsgan_loss_G(lf2_G)) * 0.5
                l_fm  = (feature_matching_loss(feats_real1, feats_fake1)
                       + feature_matching_loss(feats_real2, feats_fake2)) * 0.5
                loss_G = sup_losses["total"] + w_adv * l_adv + W_FM * l_fm

            scaler.scale(loss_G).backward()
            scaler.unscale_(optimizer_G_gan)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer_G_gan)
            scaler.update()
            if ema_active and ema is not None:
                ema.update()

            acc_G["sup"]  += sup_losses["total"].detach().item()
            acc_G["adv"]  += l_adv.detach().item()
            acc_G["fm"]   += l_fm.detach().item()
            acc_D["D"]    += loss_D.detach().item()
            acc_D["real"] += lr1.mean().detach().item()
            acc_D["fake"] += lf1.mean().detach().item()
            n_steps += 1
            step_counter += 1

        scheduler_G_gan.step()

        # ── Validation ────────────────────────────────────────────────────────
        val_metrics = validate(
            model, dl_val, lpips_eval_fn, epoch=90 + epoch_gan,
            ema=ema if ema_active else None,
            generate_panel=True,
        )

        if val_metrics["S"] < best_S_gan:
            best_S_gan = val_metrics["S"]
            save_checkpoint(CKPT_DIR / "best_gan.pt", model, optimizer_G_gan,
                    scheduler_G_gan, scaler, ema,
                    90 + epoch_gan, val_metrics, {}, full=True)

            torch.save({
                "D1": D1.state_dict(),
                "D2": D2.state_dict(),
                "optimizer_D": optimizer_D.state_dict(),
                "epoch_gan": epoch_gan,
            }, CKPT_DIR / "best_gan_disc.pt")

        save_checkpoint(CKPT_DIR / "last_gan.pt", model, optimizer_G_gan,
                    scheduler_G_gan, scaler, ema,
                    90 + epoch_gan, val_metrics, {}, full=True)

        torch.save({
            "D1": D1.state_dict(),
            "D2": D2.state_dict(),
            "optimizer_D": optimizer_D.state_dict(),
            "epoch_gan": epoch_gan,
        }, CKPT_DIR / "last_gan_disc.pt")

        # ── GAN monitoring block ──────────────────────────────────────────────
        epoch_time = time.perf_counter() - epoch_start
        mins, secs = divmod(int(epoch_time), 60)
        r_t = acc_D["real"] / n_steps
        print("=" * 72)
        print(f"GAN EPOCH {epoch_gan:03d} | {mins}m {secs:02d}s | w_adv={w_adv:.2f}")
        print(f"GENERATOR LOSSES")
        print(f"  supervised={acc_G['sup']/n_steps:.4f}  "
              f"adv={acc_G['adv']/n_steps:.4f}  "
              f"fm={acc_G['fm']/n_steps:.4f}")
        print(f"DISCRIMINATOR")
        print(f"  L_D={acc_D['D']/n_steps:.4f}  "
              f"D(real)={r_t:.3f}  "
              f"D(fake)={acc_D['fake']/n_steps:.3f}  "
              f"r_t(sign)={r_t:.2f}")
        if r_t > 0.85:
            print(f"  WARNING: D(real) > 0.85 — discriminator may be overfitting")
        print(f"VALIDATION METRICS")
        print(f"  Normal   : MAE={val_metrics['mae_normal']:.2f}°  "
              f"cos_dist={val_metrics['cos_dist']:.4f}")
        print(f"  Roughness: MAE={val_metrics['mae_roughness']:.4f}  "
              f"RMSE={val_metrics['rmse_roughness']:.4f}")
        print(f"  Render   : LPIPS={val_metrics.get('lpips_render', float('nan')):.4f}")
        print(f"  S composite: {val_metrics['S']:.4f}  (best GAN: {best_S_gan:.4f})")
        print(f"PANEL: panels/panel_ep{90+epoch_gan:03d}.png")
        print("=" * 72)

    print(f"\nGAN fine-tuning complete. Best S (GAN): {best_S_gan:.4f}")
    print(f"Checkpoints: best_gan.pt, last_gan.pt")

Supervised training complete. Skipping to GAN phase.

Starting training: epoch 90 → 89
GAN phase: loaded generator from best_overall.pt (epoch 89)
GAN phase: discriminator initialized from scratch

Starting GAN fine-tuning: 20 epochs


/tmp/ipykernel_23/4047183173.py:257: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
/tmp/ipykernel_23/4047183173.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/4047183173.py:293: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/3562339322.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/tmp/ipykernel_23/2430994994.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/2430994994.py:75: FutureWarning: `torch.cud

GAN EPOCH 000 | 6m 59s | w_adv=0.02
GENERATOR LOSSES
  supervised=0.1809  adv=0.4262  fm=0.1563
DISCRIMINATOR
  L_D=0.3465  D(real)=0.508  D(fake)=0.462  r_t(sign)=0.51
VALIDATION METRICS
  Normal   : MAE=10.47°  cos_dist=0.0419
  Roughness: MAE=0.1086  RMSE=0.1479
  Render   : LPIPS=0.1025
  S composite: 10.5544  (best GAN: 10.5544)
PANEL: panels/panel_ep090.png
GAN EPOCH 001 | 5m 22s | w_adv=0.02
GENERATOR LOSSES
  supervised=0.1808  adv=0.2768  fm=0.0735
DISCRIMINATOR
  L_D=0.2718  D(real)=0.497  D(fake)=0.496  r_t(sign)=0.50
VALIDATION METRICS
  Normal   : MAE=10.41°  cos_dist=0.0412
  Roughness: MAE=0.1095  RMSE=0.1500
  Render   : LPIPS=0.0995
  S composite: 10.4958  (best GAN: 10.4958)
PANEL: panels/panel_ep091.png
GAN EPOCH 002 | 5m 23s | w_adv=0.02
GENERATOR LOSSES
  supervised=0.1816  adv=0.2668  fm=0.0573
DISCRIMINATOR
  L_D=0.2658  D(real)=0.498  D(fake)=0.498  r_t(sign)=0.50
VALIDATION METRICS
  Normal   : MAE=10.39°  cos_dist=0.0410
  Roughness: MAE=0.1090  RMSE=0.1485
  

## Cell 13 — Post-training summary and progression GIF

In [14]:
panel_files = sorted(PANELS_DIR.glob("panel_ep*.png"))
print(f"\nTraining complete.")
print(f"Total epochs   : {TOTAL_EPOCHS}")
print(f"Best S         : {best_metrics['S']:.4f} @ epoch {best_metrics['epoch']}")
print(f"Best normal MAE: {best_metrics.get('mae_normal_best', float('nan')):.2f}°")
print(f"Panels saved   : {len(panel_files)}")
print(f"Checkpoints in : {CKPT_DIR}")
 
if len(panel_files) >= 2:
    try:
        frames = [Image.open(f).convert("RGB") for f in panel_files]
        gif_path = OUTPUT_DIR / "training_progression.gif"
        frames[0].save(
            gif_path, save_all=True, append_images=frames[1:],
            duration=800, loop=0, optimize=False,
        )
        print(f"Progression GIF: {gif_path}")
    except Exception as e:
        print(f"GIF creation failed (non-critical): {e}")
 
if not DRY_RUN:
    print("\nTraining complete. Next: matforge_04_baseline_eval")
    print("  → Evaluate DeepPBR baseline on MatForge val split")
    print("  → Build final comparison table")
else:
    print("\nDRY RUN complete. Verify the monitoring output above, then:")
    print("  1. Set DRY_RUN = False")
    print("  2. Submit the notebook for full training")


Training complete.
Total epochs   : 90
Best S         : inf @ epoch -1
Best normal MAE: nan°
Panels saved   : 20
Checkpoints in : /kaggle/working/checkpoints
Progression GIF: /kaggle/working/training_progression.gif

Training complete. Next: matforge_04_baseline_eval
  → Evaluate DeepPBR baseline on MatForge val split
  → Build final comparison table
